# Phase 3 — Feature Engineering

## 1. Load Processed Data

In [14]:
import pandas as pd

train = pd.read_csv("../data/processed/train_processed.csv")
test = pd.read_csv("../data/processed/test_processed.csv")

## 2. Area Features

In [15]:
for df in [train, test]:
    df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
    df["TotalBathrooms"] = (
        df["FullBath"] + 0.5 * df["HalfBath"] +
        df["BsmtFullBath"] + 0.5 * df["BsmtHalfBath"]
    )
    df["TotalPorchSF"] = (
        df["OpenPorchSF"] + df["3SsnPorch"] +
        df["EnclosedPorch"] + df["ScreenPorch"] + df["WoodDeckSF"]
    )
    df["TotalBsmtFinishedSF"] = df["BsmtFinSF1"] + df["BsmtFinSF2"]

## 3. Age Features

In [16]:
for df in [train, test]:
    df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
    df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]
    df["GarageAge"] = df["YrSold"] - df["GarageYrBlt"]

## 4. Quality × Size Features

In [17]:
for df in [train, test]:
    df["Qual_GrLivArea"] = df["OverallQual"] * df["GrLivArea"]
    df["Qual_TotalSF"] = df["OverallQual"] * df["TotalSF"]
    df["Qual_TotalBsmtSF"] = df["OverallQual"] * df["TotalBsmtSF"]

## 5. Additional Domain Features

In [18]:
for df in [train, test]:
    df["TotalLivingArea"] = df["GrLivArea"] + df["TotalBsmtFinishedSF"]
    df["TotalRooms"] = df["TotRmsAbvGrd"] + df["FullBath"] + df["HalfBath"]
    df["AreaPerRoom"] = df["GrLivArea"] / df["TotRmsAbvGrd"].clip(lower=1)
    df["GarageAreaPerCar"] = df["GarageArea"] / df["GarageCars"].clip(lower=1)

## 6. Final Validation

In [19]:
import numpy as np

print("Missing values:", train.isna().sum().sum(), test.isna().sum().sum())
print("Infinite values:", np.isinf(train.select_dtypes("number")).sum().sum(),
      np.isinf(test.select_dtypes("number")).sum().sum())

Missing values: 0 0
Infinite values: 0 0


## 7. Engineered Features

In [21]:
new_features = [
    "TotalSF", "TotalBathrooms", "TotalPorchSF", "TotalBsmtFinishedSF",
    "HouseAge", "RemodAge", "GarageAge",
    "Qual_GrLivArea", "Qual_TotalSF", "Qual_TotalBsmtSF",
    "TotalLivingArea", "TotalRooms", "AreaPerRoom", "GarageAreaPerCar"
]

train[new_features].describe().T

,count,mean,std,min,25%,50%,75%,max
TotalSF,1460.0,2567.048630,821.714421,334.0,2009.500000,2474.0,3004.000000,11752.000000
TotalBathrooms,1460.0,2.210616,0.785399,1.0,2.000000,2.0,2.500000,6.000000
TotalPorchSF,1460.0,181.329452,156.656097,0.0,45.000000,164.0,266.000000,1027.000000
TotalBsmtFinishedSF,1460.0,490.189041,476.103307,0.0,0.000000,465.0,790.250000,5644.000000
HouseAge,1460.0,36.547945,30.250152,0.0,8.000000,35.0,54.000000,136.000000
RemodAge,1460.0,22.950000,20.640653,-1.0,4.000000,14.0,41.000000,60.000000
GarageAge,1460.0,139.076027,453.714026,0.0,7.000000,30.0,50.000000,2010.000000
Qual_GrLivArea,1460.0,9673.956164,5186.744876,334.0,5790.000000,8820.0,12180.000000,56420.000000
Qual_TotalSF,1460.0,16416.028767,8665.496074,334.0,10425.000000,14718.0,20105.750000,117520.000000
Qual_TotalBsmtSF,1460.0,6775.675342,4116.683404,0.0,4333.500000,5656.0,8634.000000,61100.000000


In [22]:
train.loc[train["RemodAge"] < 0, ["Id", "YearRemodAdd", "YrSold", "RemodAge"]]

,Id,YearRemodAdd,YrSold,RemodAge
523,524,2008,2007,-1


In [23]:
for df in [train, test]:
    df["RemodAge"] = df["RemodAge"].clip(lower=0)

In [24]:
for df in [train, test]:
    df["GarageAge"] = df["GarageAge"].where(df["GarageYrBlt"] > 0, 0)

In [25]:
train[["RemodAge", "GarageAge"]].describe()

,RemodAge,GarageAge
count,1460.000000,1460.000000
mean,22.950685,27.680137
std,20.639875,24.950144
min,0.000000,0.000000
25%,4.000000,4.000000
50%,14.000000,23.500000
75%,41.000000,46.000000
max,60.000000,107.000000


## 8. Save Engineered Data

In [26]:
train.to_csv("../data/processed/train_engineered.csv", index=False)
test.to_csv("../data/processed/test_engineered.csv", index=False)

In [27]:
train_check = pd.read_csv("../data/processed/train_engineered.csv")
test_check = pd.read_csv("../data/processed/test_engineered.csv")

print(train_check.shape, test_check.shape)
print(train_check.isna().sum().sum(), test_check.isna().sum().sum())

(1460, 95) (1459, 94)
0 0
